# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 human protein mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Citation: {getattr(metadata, 'citeAs', '')}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets, the fields (columns) available in each, and display their `@id` for reference.

In [ ]:
# Explore record sets, fields, and columns
record_sets = dataset.metadata.record_sets

print(f"Record sets in dataset:")
for rs in record_sets:
    print(f"  - Record set name: {rs.name} | @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - Field name: {field.name} | @id: {field.id} | Data type: {getattr(field, 'data_type', 'unknown')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

To make the notebook reproducible, we will extract all record sets and show an example for the one containing the most abundant protein annotation table (main record set).

Entities in `mlcroissant` are always referenced by their `@id`.

In [ ]:
# Collect all record set @id values
rs_ids = [rs.id for rs in dataset.metadata.record_sets]
print("All record set @id values:")
print(rs_ids)

# For example, select the first record set (replace with the correct @id for your analysis)
main_record_set_id = rs_ids[0]
dataframes = {}

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id}")

print(f"\nColumns in main record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())

# Preview data from the main record set
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping by categories.

Below, we select numeric and group fields by their `@id`. Adjust field `@id` values to match those printed in the overview for specialized analysis.

In [ ]:
# Use one of the available numeric fields from the main record set
# Suppose the main protein table uses field IDs like 'cr:coverage', 'cr:mw', 'cr:abundance', etc.

# Edit these to match your dataset's field @ids (see code above for actual values)
numeric_field_id = None
group_field_id = None

# Auto-suggest likely numeric and group fields
col_names = dataframes[main_record_set_id].columns
# Try to pick well-known quantities - adjust based on actual field @ids
for cname in col_names:
    if 'coverage' in cname.lower():
        numeric_field_id = cname
    elif 'mw' in cname.lower() or 'mass' in cname.lower():
        if numeric_field_id is None:
            numeric_field_id = cname
    elif 'abundance' in cname.lower():
        if numeric_field_id is None:
            numeric_field_id = cname
    if 'description' in cname.lower() or 'annotation' in cname.lower() or 'sample' in cname.lower():
        group_field_id = cname

print(f"Chosen numeric field for analysis: {numeric_field_id}")
print(f"Chosen group field for grouping: {group_field_id}")

df = dataframes[main_record_set_id]

# Remove rows with missing numeric values
df = df.dropna(subset=[numeric_field_id])

# Ensure numeric type
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
df = df.dropna(subset=[numeric_field_id])

# Filtering records with a high numeric value threshold (e.g., coverage > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records (where {numeric_field_id} > {threshold}): {len(filtered_df)} found")
print(filtered_df.head())

# Normalizing the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If suitable, group and aggregate (requires group_field_id to be set)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index(name=f"mean_{numeric_field_id}")
    print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[numeric_field_id], bins=30)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouped, show mean by group (e.g., barplot for mean coverage by sample/description)
if group_field_id and group_field_id in filtered_df.columns:
    top = grouped_df.sort_values(f"mean_{numeric_field_id}", ascending=False).head(10)
    plt.figure(figsize=(8,4))
    sns.barplot(data=top, x=group_field_id, y=f"mean_{numeric_field_id}")
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load and inspect a Croissant dataset using `mlcroissant` and reference entities using their `@id`.
- Explore record sets, fields, and their metadata identifiers.
- Extract tabular data, filter and normalize numeric protein abundance fields, and group by categorical annotation columns.
- Visualize numeric distributions and (optionally) compare groups.

You can further extend this notebook to explore biological questions, deeper aggregations, or machine learning tasks using the curated FAIR^2 dataset.